# Phase 8 Stage 1b: Fine-tuning Only
Load pre-trained Stage 1a model and fine-tune with Frozen Qwen2-0.5B.

This notebook skips Stage 1a training (assuming phase8_training_best.pt already exists).
- Approach A/B: DifferenceModule + Adaptive Bridge → Frozen Qwen2-0.5B
- Approach C: Q-Former → Frozen Qwen2-VL-2B

Comparison & PDF report included.

In [ ]:
# Cell 1: Environment setup
import os
from pathlib import Path

import torch

from src.utils import get_device, set_seed

print("=" * 70)
print("PHASE 8 STAGE 1b: FINE-TUNING ONLY")
print("=" * 70)

set_seed(42)
device = get_device()
print(f"[Phase 8] Compute device: {device}")
if device.type == "cuda":
    print(f"[Phase 8] GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[Phase 8] Running on CPU.")

CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 2: Load config and data loaders
from src.config import DataConfig
from src.dataset import build_remoteclip_transforms, get_levircc_loaders, get_secondcc_loaders
from src.models.phase8 import Phase8Config

data_config = DataConfig()
phase8_config = Phase8Config()

BATCH_SIZE = 8
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4

remoteclip_transforms = build_remoteclip_transforms(
    img_size=phase8_config.img_size,
    mean=phase8_config.remoteclip_mean,
    std=phase8_config.remoteclip_std,
)

train_loader, val_loader, test_loader, vocab = get_levircc_loaders(
    caption_json=data_config.caption_json,
    image_root=data_config.image_root,
    batch_size=BATCH_SIZE,
    device=str(device),
    img_size=phase8_config.img_size,
    num_workers=data_config.num_workers,
    vocab_path=data_config.vocab_path,
    transforms_fn=remoteclip_transforms,
)

# Try to load SECOND-CC if available
SECONDCC_CAPTION_JSON = Path("SECOND-CC-AUG/SECOND-CC-AUG.json")
SECONDCC_IMAGE_ROOT = Path("SECOND-CC-AUG")

if SECONDCC_CAPTION_JSON.is_file():
    _, _, secondcc_test_loader, _ = get_secondcc_loaders(
        caption_json=SECONDCC_CAPTION_JSON,
        image_root=SECONDCC_IMAGE_ROOT,
        vocab=vocab,
        batch_size=BATCH_SIZE,
        device=str(device),
        img_size=phase8_config.img_size,
        num_workers=data_config.num_workers,
        transforms_fn=remoteclip_transforms,
    )
    print(f"[Phase 8] SECOND-CC loaded - test batches: {len(secondcc_test_loader)}")
else:
    secondcc_test_loader = None
    print(f"[Phase 8] SECOND-CC not found at {SECONDCC_CAPTION_JSON}")

print(f"[Phase 8] Vocabulary size: {len(vocab.word2idx)}")
print(f"[Phase 8] LEVIR-CC train/val/test batches: {len(train_loader)}/{len(val_loader)}/{len(test_loader)}")

In [ ]:
# Cell 3: Load pre-trained Stage 1a model
from src.models.phase8 import DifferenceRSICCModel
from src.training import load_checkpoint
from src.metrics import SentenceEmbeddingScorer
from src.dataset import Vocabulary

print("\n" + "=" * 70)
print("LOADING PRE-TRAINED STAGE 1a MODEL")
print("=" * 70)

# Build and load Stage 1a model
s1a_model = DifferenceRSICCModel(vocab=vocab, config=phase8_config)
s1a_model.to(device)

best_training_ckpt = CHECKPOINT_DIR / "phase8_training_best.pt"
if best_training_ckpt.is_file():
    # Add safe globals for PyTorch 2.6+ security
    torch.serialization.add_safe_globals([Vocabulary])
    ckpt = torch.load(best_training_ckpt, map_location=device, weights_only=False)
    s1a_model.load_state_dict(ckpt["model_state_dict"])
    loaded_epoch = ckpt.get("epoch", "?")
    loaded_val_loss = ckpt.get("val_loss", float("inf"))
    print(f"[Phase 8] Loaded Stage 1a model from epoch {loaded_epoch} (val_loss={loaded_val_loss:.4f})")
    print(f"[Phase 8] Checkpoint: {best_training_ckpt}")
else:
    print(f"[ERROR] Stage 1a checkpoint not found at {best_training_ckpt}")
    print(f"[ERROR] Please run phase8.ipynb (training) first or check checkpoint path.")
    raise FileNotFoundError(f"Missing checkpoint: {best_training_ckpt}")

# Initialize semantic scorer for evaluation
semantic_scorer = SentenceEmbeddingScorer()

In [ ]:
# Cell 4: Import training utilities
from src.training import train_epoch, validate
from src.metrics import evaluate_full_test_with_per_sample_metrics, save_phase1_results

print("[Phase 8] Training utilities loaded")

In [ ]:
# Cell 5a: Fine-tuning Approach A/B - DifferenceModule + Bridge (FROZEN QWEN)
print("\n" + "=" * 70)
print("PHASE 8 STAGE 1b - APPROACH A/B: DifferenceModule + Bridge → Frozen Qwen")
print("=" * 70)

import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

# Approach A/B: Train ONLY bridge, keep Qwen frozen
class Phase8HybridModelAB(nn.Module):
    """DifferenceModule + Adaptive bridge to FROZEN Qwen2-0.5B LLM."""
    
    def __init__(self, trained_model, llm_model_id="Qwen/Qwen2-0.5B-Instruct"):
        super().__init__()
        self.encoder = trained_model.encoder
        self.diff_module = trained_model.diff_module
        
        # Load Qwen2-0.5B with float16 (no quantization needed - plenty of VRAM)
        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_model_id,
            torch_dtype=torch.float16,
            trust_remote_code=True,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(llm_model_id, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # Adaptive bridge: Create directly in float16 (not float32 then convert)
        self.diff_to_llm = nn.Sequential(
            nn.Linear(phase8_config.fusion_dim, 1024, dtype=torch.float16),
            nn.LayerNorm(1024, dtype=torch.float16),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, self.llm.config.hidden_size, dtype=torch.float16),
            nn.LayerNorm(self.llm.config.hidden_size, dtype=torch.float16),
        )
        
        if hasattr(self.llm, "visual"):
            self.llm.visual = None
        
        print("[Phase 8] Approach A/B Model - DifferenceModule + Adaptive Bridge:")
        print(f"  ✓ Encoder: RemoteCLIP (frozen from Stage 1a)")
        print(f"  ✓ DifferenceModule: TRAINABLE (learn change representation)")
        print(f"  ✓ Adaptive Bridge: TRAINABLE (float16, connect to Qwen)")
        print(f"  ✓ LLM: Qwen2-0.5B float16 (FROZEN - only bridge adapts)")
    
    def forward(self, before_image, after_image, input_ids, attention_mask=None, labels=None):
        # Encode (frozen - no gradient computation)
        with torch.no_grad():
            before_feat = self.encoder(before_image)
            after_feat = self.encoder(after_image)
            if before_feat.dim() == 2:
                before_feat = before_feat.unsqueeze(1)
                after_feat = after_feat.unsqueeze(1)
        
        # DifferenceModule (trainable)
        diff_embedding = self.diff_module(before_feat, after_feat)
        
        # Adaptive bridge (trainable - adapts difference features to Qwen space, in float16)
        llm_prefix = self.diff_to_llm(diff_embedding.to(torch.float16))
        
        # Text embeddings (from frozen Qwen in float16, no gradient computation)
        with torch.no_grad():
            text_embeddings = self.llm.get_input_embeddings()(input_ids)
        
        # Concatenate visual prefix + text (both in float16)
        combined_embeds = torch.cat([llm_prefix, text_embeddings], dim=1)
        
        # Attention mask (ensure correct dtype for concatenation)
        visual_attn = torch.ones(before_image.size(0), diff_embedding.size(1), 
                                 device=input_ids.device, dtype=torch.float)
        # Convert attention_mask to float if needed
        if attention_mask.dtype != torch.float:
            attention_mask = attention_mask.float()
        combined_attn = torch.cat([visual_attn, attention_mask], dim=1)
        
        # Labels: prepend -100 (ignore_index) for visual prefix since no ground truth
        combined_labels = None
        if labels is not None:
            visual_labels = torch.full(
                (labels.size(0), diff_embedding.size(1)),
                -100,  # ignore_index - don't compute loss for visual part
                dtype=labels.dtype,
                device=labels.device
            )
            combined_labels = torch.cat([visual_labels, labels], dim=1)
        
        # Forward through FROZEN Qwen (LLM params frozen, but loss computation requires gradients)
        outputs = self.llm(
            inputs_embeds=combined_embeds,
            attention_mask=combined_attn,
            labels=combined_labels,
        )
        return outputs

# Create Approach A/B model
model_ab = Phase8HybridModelAB(s1a_model)
model_ab.to(device)

# Freeze encoder and Qwen - only train DifferenceModule + Bridge
for param in model_ab.encoder.parameters():
    param.requires_grad = False
for param in model_ab.llm.parameters():
    param.requires_grad = False

trainable_ab = sum(p.numel() for p in model_ab.parameters() if p.requires_grad)
frozen_ab = sum(p.numel() for p in model_ab.parameters() if not p.requires_grad)
print(f"\n[Phase 8] Approach A/B - Training Strategy:")
print(f"  Trainable: {trainable_ab:,} (DifferenceModule + Bridge only)")
print(f"  Frozen: {frozen_ab:,} (RemoteCLIP + Qwen2-0.5B)")

# Training setup - LOWER learning rate for float16 stability
FT_EPOCHS = 10
FT_LEARNING_RATE = 5e-5
FT_WEIGHT_DECAY = 1e-5

optimizer_ab = torch.optim.Adam(
    [p for p in model_ab.parameters() if p.requires_grad],
    lr=FT_LEARNING_RATE,
    weight_decay=FT_WEIGHT_DECAY
)

# Train Approach A/B
history_ab = []
best_loss_ab = float("inf")
best_epoch_ab = 0

print(f"\n[Phase 8] Training Approach A/B (DifferenceModule + Bridge → Frozen Qwen)...")
print(f"[Phase 8] Learning Rate: {FT_LEARNING_RATE} (reduced for float16 stability)")
print(f"[Phase 8] Using GradScaler for mixed precision training")
for epoch in range(1, FT_EPOCHS + 1):
    train_loss = train_epoch(
        model_ab, train_loader, optimizer_ab, device,
        grad_clip=1.0, log_every=10, max_caption_len=phase8_config.max_caption_len
    )
    val_loss = validate(model_ab, val_loader, device, max_caption_len=phase8_config.max_caption_len)
    
    history_ab.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    print(f"Epoch {epoch}/{FT_EPOCHS}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
    
    if val_loss < best_loss_ab:
        best_loss_ab = val_loss
        best_epoch_ab = epoch
        ckpt_ab_best = CHECKPOINT_DIR / "phase8_finetune_AB_best.pt"
        torch.save({
            "model_state_dict": model_ab.state_dict(),
            "optimizer_state_dict": optimizer_ab.state_dict(),
            "epoch": epoch,
            "vocab": vocab,
            "val_loss": val_loss,
            "approach": "A/B_DifferenceModule_FrozenQwen",
        }, ckpt_ab_best)
        print(f"  ✓ Saved best A/B checkpoint (val_loss={val_loss:.4f})")

ckpt_ab_current = CHECKPOINT_DIR / "phase8_finetune_AB_current.pt"
torch.save({
    "model_state_dict": model_ab.state_dict(),
    "optimizer_state_dict": optimizer_ab.state_dict(),
    "epoch": epoch,
    "vocab": vocab,
    "val_loss": val_loss,
    "approach": "A/B_DifferenceModule_FrozenQwen",
}, ckpt_ab_current)

print(f"\n✓ Approach A/B complete - Best epoch: {best_epoch_ab} (val_loss={best_loss_ab:.4f})")

In [ ]:
# Cell 5b: Fine-tuning Approach C - Q-Former → Frozen Qwen2-VL (Only Q-Former Trained)
print("\n" + "=" * 70)
print("PHASE 8 STAGE 1b - APPROACH C: Q-Former → Frozen Qwen2-VL")
print("=" * 70)

from src.models.codeaug import CodeAugRSICCModel

try:
    # Approach C: Use CodeAugRSICCModel but freeze Qwen
    model_c = CodeAugRSICCModel(vocab=vocab)
    model_c.to(device)
    
    # Ensure LLM is in float16 if it exists
    if hasattr(model_c, 'llm') and model_c.llm is not None:
        model_c.llm = model_c.llm.to(torch.float16)
    
    # Freeze everything except Q-Former
    for name, param in model_c.named_parameters():
        if "qformer" in name.lower():
            param.requires_grad = True  # Q-Former is trainable
        else:
            param.requires_grad = False  # Everything else frozen
    
    # Explicitly freeze Qwen (LLM)
    if hasattr(model_c, 'llm') and model_c.llm is not None:
        for param in model_c.llm.parameters():
            param.requires_grad = False
    
    trainable_c = sum(p.numel() for p in model_c.parameters() if p.requires_grad)
    frozen_c = sum(p.numel() for p in model_c.parameters() if not p.requires_grad)
    print(f"\n[Phase 8] Approach C - Training Strategy:")
    print(f"  Trainable: {trainable_c:,} (Q-Former token compressor only)")
    print(f"  Frozen: {frozen_c:,} (RemoteCLIP ViT-L-14 + Qwen2-VL)")
    
    # Training setup
    optimizer_c = torch.optim.Adam(
        [p for p in model_c.parameters() if p.requires_grad],
        lr=FT_LEARNING_RATE,
        weight_decay=FT_WEIGHT_DECAY
    )
    
    # Train Approach C
    history_c = []
    best_loss_c = float("inf")
    best_epoch_c = 0
    
    print(f"\n[Phase 8] Training Approach C (Q-Former → Frozen Qwen2-VL)...")
    for epoch in range(1, FT_EPOCHS + 1):
        train_loss = train_epoch(
            model_c, train_loader, optimizer_c, device,
            grad_clip=1.0, log_every=10, max_caption_len=phase8_config.max_caption_len
        )
        val_loss = validate(model_c, val_loader, device, max_caption_len=phase8_config.max_caption_len)
        
        history_c.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"Epoch {epoch}/{FT_EPOCHS}: train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
        
        if val_loss < best_loss_c:
            best_loss_c = val_loss
            best_epoch_c = epoch
            ckpt_c_best = CHECKPOINT_DIR / "phase8_finetune_C_best.pt"
            torch.save({
                "model_state_dict": model_c.state_dict(),
                "optimizer_state_dict": optimizer_c.state_dict(),
                "epoch": epoch,
                "vocab": vocab,
                "val_loss": val_loss,
                "approach": "C_QFormer_FrozenQwen",
            }, ckpt_c_best)
            print(f"  ✓ Saved best C checkpoint (val_loss={val_loss:.4f})")
    
    ckpt_c_current = CHECKPOINT_DIR / "phase8_finetune_C_current.pt"
    torch.save({
        "model_state_dict": model_c.state_dict(),
        "optimizer_state_dict": optimizer_c.state_dict(),
        "epoch": epoch,
        "vocab": vocab,
        "val_loss": val_loss,
        "approach": "C_QFormer_FrozenQwen",
    }, ckpt_c_current)
    
    print(f"\n✓ Approach C complete - Best epoch: {best_epoch_c} (val_loss={best_loss_c:.4f})")
    
except Exception as e:
    print(f"\n[ERROR] Approach C failed: {str(e)}")
    print(f"[WARNING] Skipping Approach C - will only evaluate Approach A/B")
    model_c = None
    history_c = []
    best_loss_c = float("inf")
    best_epoch_c = 0
    ckpt_c_best = None

print("\n" + "=" * 70)
print("TRAINING COMPLETE - PREPARING EVALUATION")
print("=" * 70)

In [ ]:
# Cell 6a: Evaluate Approach A/B on LEVIR-CC
print("\n" + "=" * 70)
print("EVALUATING APPROACH A/B (DifferenceModule + Adaptive Bridge)")
print("=" * 70)

# Load best A/B checkpoint
ckpt_ab_best = CHECKPOINT_DIR / "phase8_finetune_AB_best.pt"
if ckpt_ab_best.is_file():
    # Add safe globals for PyTorch 2.6+ security
    torch.serialization.add_safe_globals([Vocabulary])
    ckpt_data = torch.load(ckpt_ab_best, map_location=device, weights_only=False)
    model_ab.load_state_dict(ckpt_data["model_state_dict"])
    model_ab.to(device)
    model_ab.eval()
    
    epoch_ab = ckpt_data.get("epoch", "?")
    loss_ab = ckpt_data.get("val_loss", float("inf"))
    print(f"[Phase 8] Loaded best A/B model (epoch {epoch_ab}, val_loss={loss_ab:.4f})")
else:
    print(f"[Phase 8] ERROR: A/B checkpoint not found")
    epoch_ab = "?"
    loss_ab = float("inf")

# Evaluate on LEVIR-CC
metrics_ab_levir, samples_ab_levir = evaluate_full_test_with_per_sample_metrics(
    model_ab, test_loader, device,
    semantic_model=semantic_scorer,
    max_new_tokens=phase8_config.max_caption_len,
)

print("\n" + "=" * 70)
print("APPROACH A/B - LEVIR-CC TEST RESULTS")
print("=" * 70)
for k, v in metrics_ab_levir.items():
    print(f"  {k:<25}: {v:.4f}" if isinstance(v, float) else f"  {k:<25}: {v}")

save_phase1_results(
    CHECKPOINT_DIR / "phase8_finetune_AB_levircc_test_results.json",
    metrics_ab_levir, samples_ab_levir,
    metadata={"dataset": "LEVIR-CC", "approach": "A/B_DifferenceModule", "checkpoint_epoch": epoch_ab},
)

In [ ]:
# Cell 6b: Evaluate Approach C on LEVIR-CC
metrics_c_levir, samples_c_levir = None, None
if model_c is not None:
    print("\n" + "=" * 70)
    print("EVALUATING APPROACH C (Q-Former + Qwen2-VL)")
    print("=" * 70)
    
    # Load best C checkpoint
    ckpt_c_best = CHECKPOINT_DIR / "phase8_finetune_C_best.pt"
    if ckpt_c_best.is_file():
        # Add safe globals for PyTorch 2.6+ security
        torch.serialization.add_safe_globals([Vocabulary])
        ckpt_data = torch.load(ckpt_c_best, map_location=device, weights_only=False)
        model_c.load_state_dict(ckpt_data["model_state_dict"])
        model_c.to(device)
        model_c.eval()
        
        epoch_c = ckpt_data.get("epoch", "?")
        loss_c = ckpt_data.get("val_loss", float("inf"))
        print(f"[Phase 8] Loaded best C model (epoch {epoch_c}, val_loss={loss_c:.4f})")
    else:
        print(f"[Phase 8] ERROR: C checkpoint not found")
        epoch_c = "?"
        loss_c = float("inf")
        metrics_c_levir = None
        samples_c_levir = None
    
    # Evaluate on LEVIR-CC only if model loaded successfully
    if metrics_c_levir is None:
        try:
            metrics_c_levir, samples_c_levir = evaluate_full_test_with_per_sample_metrics(
                model_c, test_loader, device,
                semantic_model=semantic_scorer,
                max_new_tokens=phase8_config.max_caption_len,
            )
            
            print("\n" + "=" * 70)
            print("APPROACH C - LEVIR-CC TEST RESULTS")
            print("=" * 70)
            for k, v in metrics_c_levir.items():
                print(f"  {k:<25}: {v:.4f}" if isinstance(v, float) else f"  {k:<25}: {v}")
            
            save_phase1_results(
                CHECKPOINT_DIR / "phase8_finetune_C_levircc_test_results.json",
                metrics_c_levir, samples_c_levir,
                metadata={"dataset": "LEVIR-CC", "approach": "C_QFormer", "checkpoint_epoch": epoch_c},
            )
        except Exception as e:
            print(f"[ERROR] Failed to evaluate Approach C: {str(e)}")
            metrics_c_levir = None
            samples_c_levir = None
else:
    print("\n[WARNING] Approach C was skipped - no evaluation results available")
    metrics_c_levir = None
    samples_c_levir = None
    epoch_c = "N/A"
    loss_c = float("inf")

In [ ]:
# Cell 7: Evaluate both approaches on SECOND-CC (Transfer Learning)
if secondcc_test_loader is not None:
    print("\n" + "=" * 70)
    print("EVALUATING BOTH APPROACHES ON SECOND-CC (Transfer Learning)")
    print("=" * 70)
    
    # Approach A/B on SECOND-CC
    print("\nApproach A/B on SECOND-CC...")
    try:
        metrics_ab_secondcc, samples_ab_secondcc = evaluate_full_test_with_per_sample_metrics(
            model_ab, secondcc_test_loader, device,
            semantic_model=semantic_scorer,
            max_new_tokens=phase8_config.max_caption_len,
        )
        
        print("APPROACH A/B - SECOND-CC TEST RESULTS")
        for k, v in metrics_ab_secondcc.items():
            print(f"  {k:<25}: {v:.4f}" if isinstance(v, float) else f"  {k:<25}: {v}")
        
        save_phase1_results(
            CHECKPOINT_DIR / "phase8_finetune_AB_secondcc_test_results.json",
            metrics_ab_secondcc, samples_ab_secondcc,
            metadata={"dataset": "SECOND-CC", "approach": "A/B_DifferenceModule", "checkpoint_epoch": epoch_ab},
        )
    except Exception as e:
        print(f"[ERROR] Failed to evaluate Approach A/B on SECOND-CC: {str(e)}")
        metrics_ab_secondcc = None
        samples_ab_secondcc = None
    
    # Approach C on SECOND-CC (if C model exists)
    if model_c is not None:
        print("\nApproach C on SECOND-CC...")
        try:
            metrics_c_secondcc, samples_c_secondcc = evaluate_full_test_with_per_sample_metrics(
                model_c, secondcc_test_loader, device,
                semantic_model=semantic_scorer,
                max_new_tokens=phase8_config.max_caption_len,
            )
            
            print("APPROACH C - SECOND-CC TEST RESULTS")
            for k, v in metrics_c_secondcc.items():
                print(f"  {k:<25}: {v:.4f}" if isinstance(v, float) else f"  {k:<25}: {v}")
            
            save_phase1_results(
                CHECKPOINT_DIR / "phase8_finetune_C_secondcc_test_results.json",
                metrics_c_secondcc, samples_c_secondcc,
                metadata={"dataset": "SECOND-CC", "approach": "C_QFormer", "checkpoint_epoch": epoch_c},
            )
        except Exception as e:
            print(f"[ERROR] Failed to evaluate Approach C on SECOND-CC: {str(e)}")
            metrics_c_secondcc = None
            samples_c_secondcc = None
    else:
        print("\n[WARNING] Approach C not available - skipping SECOND-CC evaluation")
        metrics_c_secondcc = None
        samples_c_secondcc = None
else:
    metrics_ab_secondcc = None
    metrics_c_secondcc = None
    print("[Phase 8] SECOND-CC not found - skipping transfer learning evaluation")

In [ ]:
# Cell 8: Final Comparison Report - A/B vs C
print("\n\n" + "=" * 120)
print("PHASE 8 FINAL COMPARISON: APPROACH A/B (DifferenceModule) vs APPROACH C (Q-Former)")
print("=" * 120)

# Check if both approaches have metrics
if metrics_c_levir is None:
    print("\n[WARNING] Approach C metrics not available - showing Approach A/B results only")
    print("=" * 120)
    print("\nAPPROACH A/B - LEVIR-CC TEST RESULTS")
    print("=" * 120)
    for k, v in metrics_ab_levir.items():
        print(f"  {k:<25}: {v:.4f}" if isinstance(v, float) else f"  {k:<25}: {v}")
else:
    # LEVIR-CC Comparison
    print("\n" + "-" * 120)
    print("LEVIR-CC TEST SET COMPARISON")
    print("-" * 120)
    print(f"{'METRIC':<25}{'Approach A/B (DiffModule)':<35}{'Approach C (Q-Former)':<35}{'Winner':<20}")
    print("-" * 120)
    
    winners_ab = 0
    winners_c = 0
    
    for key in sorted(metrics_ab_levir.keys()):
        if isinstance(metrics_ab_levir.get(key), float) and isinstance(metrics_c_levir.get(key), float):
            val_ab = metrics_ab_levir[key]
            val_c = metrics_c_levir[key]
            
            # Determine winner (higher is better for most metrics)
            if val_ab > val_c:
                winner = "A/B ✓"
                winners_ab += 1
            elif val_c > val_ab:
                winner = "C ✓"
                winners_c += 1
            else:
                winner = "Tie"
            
            print(f"{key:<25}{val_ab:>34.4f}{val_c:>34.4f}{winner:>20}")
    
    print(f"\nLEVIR-CC Score: Approach A/B: {winners_ab} | Approach C: {winners_c}")
    
    # SECOND-CC Comparison (if available)
    if metrics_ab_secondcc and metrics_c_secondcc:
        print("\n" + "-" * 120)
        print("SECOND-CC TEST SET COMPARISON (Transfer Learning)")
        print("-" * 120)
        print(f"{'METRIC':<25}{'Approach A/B (DiffModule)':<35}{'Approach C (Q-Former)':<35}{'Winner':<20}")
        print("-" * 120)
        
        winners_ab_2cc = 0
        winners_c_2cc = 0
        
        for key in sorted(metrics_ab_secondcc.keys()):
            if isinstance(metrics_ab_secondcc.get(key), float) and isinstance(metrics_c_secondcc.get(key), float):
                val_ab = metrics_ab_secondcc[key]
                val_c = metrics_c_secondcc[key]
                
                if val_ab > val_c:
                    winner = "A/B ✓"
                    winners_ab_2cc += 1
                elif val_c > val_ab:
                    winner = "C ✓"
                    winners_c_2cc += 1
                else:
                    winner = "Tie"
                
                print(f"{key:<25}{val_ab:>34.4f}{val_c:>34.4f}{winner:>20}")
        
        print(f"\nSECOND-CC Score: Approach A/B: {winners_ab_2cc} | Approach C: {winners_c_2cc}")

# Summary
print("\n" + "=" * 120)
print("CHECKPOINT SUMMARY")
print("=" * 120)

print("\nAPPROACH A/B (DifferenceModule + Adaptive Bridge):")
print(f"  Best Epoch: {epoch_ab}")
print(f"  Validation Loss: {loss_ab:.4f}")
print(f"  Architecture: RemoteCLIP (frozen) → DifferenceModule (trainable) → Adaptive Bridge → Frozen Qwen2-0.5B")

if model_c is not None:
    print("\nAPPROACH C (Q-Former + Qwen2-VL):")
    print(f"  Best Epoch: {epoch_c}")
    print(f"  Validation Loss: {loss_c:.4f}")
    print(f"  Architecture: RemoteCLIP ViT-L-14 (frozen) → Q-Former (trainable) → Frozen Qwen2-VL-2B")
else:
    print("\nAPPROACH C: SKIPPED (see errors above)")

print("\nRESULT FILES:")
print(f"  A/B LEVIR-CC: {CHECKPOINT_DIR / 'phase8_finetune_AB_levircc_test_results.json'}")
if metrics_c_levir is not None:
    print(f"  C LEVIR-CC:   {CHECKPOINT_DIR / 'phase8_finetune_C_levircc_test_results.json'}")
if metrics_ab_secondcc:
    print(f"  A/B SECOND-CC: {CHECKPOINT_DIR / 'phase8_finetune_AB_secondcc_test_results.json'}")
if metrics_c_secondcc:
    print(f"  C SECOND-CC:   {CHECKPOINT_DIR / 'phase8_finetune_C_secondcc_test_results.json'}")

print("\n" + "=" * 120)
if metrics_c_levir is None:
    print("✓ Phase 8 fine-tuning complete - Approach A/B results available")
elif winners_ab > winners_c:
    print("🏆 RECOMMENDATION: APPROACH A/B (DifferenceModule) performs better!")
elif winners_c > winners_ab:
    print("🏆 RECOMMENDATION: APPROACH C (Q-Former) performs better!")
else:
    print("🏆 RECOMMENDATION: Both approaches are comparable - choose based on trade-offs")
print("=" * 120)

print("\n✓ Phase 8 evaluation complete!")